In [1]:
import warnings, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer,)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import Dataset

import transformers, accelerate, peft
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("accelerate   :", accelerate.__version__)
print("peft         :", peft.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 support :", torch.cuda.is_bf16_supported())

torch        : 2.12.1+cu130
transformers : 5.12.1
accelerate   : 1.14.0
peft         : 0.19.1
CUDA         : True
GPU          : NVIDIA RTX A4000
VRAM (GB)    : 16.7
bf16 support : True


In [15]:
SEED        = 42
MODEL_NAME  = "Qwen/Qwen3-1.7B"
MAX_LEN     = 128

BATCH_SIZE            = 8
GRAD_ACCUM_STEPS      = 2
NUM_EPOCHS            = 3           
LR                    = 2e-4        
WARMUP_RATIO          = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = False        # keep compute in bf16, consistent with the DeBERTa run
print(f"Compute dtype: {'bf16' if USE_BF16 else 'fp32'}")


##ANNOTATED_FILE = Path("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample_annotated.csv")
ANNOTATED_FILE = Path("/scratch/kk01697/data/processed/annotation_sample_annotated.csv")

OUTPUT_DIR = Path("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/scripts/objective1/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "qwen3_lora_classifier"

CUSTOM_DIM = ["Narrative Structure & Quality","Character & Emotion", "Originality", "Immersion","Thematic Depth", "Writing Style",]

Compute dtype: bf16


In [3]:
ann_df = pd.read_csv(ANNOTATED_FILE)
ann_df = ann_df.dropna(subset=["sentence"])

print("Shape:", ann_df.shape)
ann_df.head()

Shape: (3000, 10)


,review_id,sentence_idx,sentence,language,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
0,6c98fe733ae0c27671ebbbe68b77fd8f,6,The initial deepening of the mechanics of the ...,eng,0,0,0,1,0,0
1,cd8abbbf2727515f904a6b189cb0eb84,24,I think that's more troubling when it comes to...,eng,0,0,0,0,0,0
2,b0d8887563f48d59440cd78144ef23c0,59,The one note simple tone of everything leads m...,en-US,0,0,0,0,0,1
3,93060ddc1ef84b111915ed91cfc443de,35,Saving grace was that he was the only characte...,eng,0,1,0,0,0,0
4,4fac8a39591a791eb0a78c4c08176d2b,4,I've never been so disappointed by this author.,eng,0,0,0,0,0,0


In [4]:
class SentenceDataset(Dataset):
    def __init__(self, texts: list[str], labels: np.ndarray, tokenizer):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

In [5]:
class NaNGuardTrainer(Trainer):
    def __init__(self, *args, pos_weight: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pos_weight = pos_weight.to(DEVICE) if pos_weight is not None else None
        self._nan_batches = 0

    # ── custom loss ───────────────────────────────────────────────────────────
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits.float()      # force fp32 even in bf16 runs
        labels  = labels.float()

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=self._pos_weight)
        loss    = loss_fn(logits, labels)

        # detect and report NaN loss without crashing
        if torch.isnan(loss):
            self._nan_batches += 1
            print(f"[NaNGuard] NaN loss detected (batch #{self._nan_batches}) — skipping")
            loss = torch.tensor(0.0, requires_grad=True, device=DEVICE)

        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        loss = super().training_step(model, inputs, num_items_in_batch)

        # zero any NaN/Inf gradients before the optimizer touches them
        # (with LoRA this only scans the small set of trainable adapter params)
        nan_params = []
        for name, param in model.named_parameters():
            if param.requires_grad and param.grad is not None:
                bad = torch.isnan(param.grad) | torch.isinf(param.grad)
                if bad.any():
                    param.grad[bad] = 0.0
                    nan_params.append(name)
        if nan_params:
            print(f"[NaNGuard] Zeroed NaN/Inf grads in: {nan_params[:3]}{'...' if len(nan_params)>3 else ''}")

        return loss

In [6]:
def build_compute_metrics(threshold: float = 0.5):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = 1.0 / (1.0 + np.exp(-logits))    # sigmoid
        preds = (probs >= threshold).astype(int)

        micro = f1_score(labels, preds, average="micro", zero_division=0)
        macro = f1_score(labels, preds, average="macro", zero_division=0)
        per   = f1_score(labels, preds, average=None,    zero_division=0)

        metrics = {"f1_micro": micro, "f1_macro": macro}
        for dim, score in zip(CUSTOM_DIM, per):
            metrics[f"f1_{dim}"] = score
        return metrics

    return compute_metrics

In [7]:
torch.manual_seed(SEED)
np.random.seed(SEED)

sentences = ann_df["sentence"].tolist()
labels    = ann_df[CUSTOM_DIM].values.astype(np.float32)

idx = np.arange(len(sentences))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=SEED)
idx_train, idx_val     = train_test_split(idx_trainval, test_size=0.15, random_state=SEED)

train_labels = labels[idx_train]
pos_counts   = train_labels.sum(axis=0).clip(min=1)
neg_counts   = len(train_labels) - pos_counts
pos_weight   = torch.tensor(neg_counts / pos_counts, dtype=torch.float32)

print(f"Split — train: {len(idx_train)}  val: {len(idx_val)}  test: {len(idx_test)}\n")
print(f"{'Dimension':<35} {'Pos':>5}  {'pos_weight':>10}")
print("-" * 55)
for dim, p, w in zip(CUSTOM_DIM, pos_counts.astype(int), pos_weight.tolist()):
    print(f"{dim:<35} {p:>5}  {w:>10.1f}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_ds = SentenceDataset([sentences[i] for i in idx_train], train_labels,      tokenizer)
val_ds   = SentenceDataset([sentences[i] for i in idx_val],   labels[idx_val],   tokenizer)
test_ds  = SentenceDataset([sentences[i] for i in idx_test],  labels[idx_test],  tokenizer)

print("Datasets created.")
print("Pad token:", tokenizer.pad_token, "| id:", tokenizer.pad_token_id)
print("Sample labels:", train_ds[0]["labels"])

Split — train: 2167  val: 383  test: 450

Dimension                             Pos  pos_weight
-------------------------------------------------------
Narrative Structure & Quality         335         5.5
Character & Emotion                   417         4.2
Originality                            80        26.1
Immersion                              66        31.8
Thematic Depth                         54        39.1
Writing Style                         125        16.3


Datasets created.
Pad token: <|endoftext|> | id: 151643
Sample labels: tensor([0., 0., 0., 0., 0., 0.])


In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CUSTOM_DIM),
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()
model.enable_input_require_grads() 

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    # the classification head is randomly initialised and small; train it in full
    modules_to_save=["score"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model on   :", next(model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[transformers] Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 17,444,864 || all params: 1,738,032,128 || trainable%: 1.0037
Model on   : cuda:0


In [9]:
_sample = {k: v.unsqueeze(0).to(DEVICE) for k, v in train_ds[0].items() if k != "labels"}
_label  = train_ds[0]["labels"].unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    _out = model(**_sample)
model.train()

_loss_fn    = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
_check_loss = _loss_fn(_out.logits.float(), _label.float())

print(f"Logits : {_out.logits.float().cpu()}")
print(f"Loss   : {_check_loss.item():.4f}")

assert not torch.isnan(_check_loss), "NaN loss before training starts — check data!"
assert _check_loss.item() < 20,      "Loss is unreasonably large — check pos_weight!"
print("\nSanity check passed \u2713")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Logits : tensor([[ 2.9688,  0.3789, -1.9219, -2.0469, -1.2031, -2.9844]])
Loss   : 0.7482

Sanity check passed ✓


In [10]:
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM_STEPS))
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)
print(f"Total optimizer steps: {total_steps}  |  Warmup steps: {warmup_steps}")

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=warmup_steps,

    fp16=False,
    bf16=USE_BF16,

    # ── gradient / memory stability ─────────────────────────────────────────
    max_grad_norm=0.5,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # ── optimizer ─────────────────────────────────────────────────────────────
    # paged_adamw_8bit keeps optimizer state memory down, important with 14B params
    optim="paged_adamw_8bit",

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    label_names=["labels"],
    save_total_limit=1,
    report_to="none",
)

Total optimizer steps: 408  |  Warmup steps: 40


In [11]:
trainer = NaNGuardTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=build_compute_metrics(),
    pos_weight=pos_weight,
)

trainer.train()
print(f"\nNaN batches caught by guard: {trainer._nan_batches}")

# saves only the LoRA adapter + classification head
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print("Adapter + tokenizer saved to", MODEL_DIR)

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
1,2.636872,1.759965,0.424581,0.313025,0.478261,0.512821,0.181818,0.177778,0.000000,0.527473
2,1.199188,1.159019,0.593592,0.508077,0.805556,0.656085,0.329114,0.270270,0.305085,0.682353
3,0.283662,3.140257,0.686636,0.534511,0.828571,0.688742,0.384615,0.307692,0.173913,0.823529



NaN batches caught by guard: 0
Adapter + tokenizer saved to /user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/scripts/outputs/qwen3_lora_classifier


In [12]:
test_results = trainer.evaluate(test_ds)

print("\n\u2500\u2500 Test-set evaluation \u2500\u2500")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro,F1 Narrative structure & quality,F1 Character & emotion,F1 Originality,F1 Immersion,F1 Thematic depth,F1 Writing style
0.283662,2.029815,3,0.717622,0.644663,0.815287,0.702381,0.521739,0.619048,0.476190,0.733333



── Test-set evaluation ──
  eval_loss: 2.0298
  eval_f1_micro: 0.7176
  eval_f1_macro: 0.6447
  eval_f1_Narrative Structure & Quality: 0.8153
  eval_f1_Character & Emotion: 0.7024
  eval_f1_Originality: 0.5217
  eval_f1_Immersion: 0.6190
  eval_f1_Thematic Depth: 0.4762
  eval_f1_Writing Style: 0.7333
